# Gastrula-to-Pup data download and preparation

This notebook downloads and prepares the **Qiu et al. 2024** mouse prenatal development dataset
([Nature 2024](https://www.nature.com/articles/s41586-024-07069-w)) for multi-modal analysis
with `RegularizedMultimodalVI`.

**Dataset summary**:
- **Protocol**: Optimised sci-RNA-seq3 (single-nucleus, combinatorial indexing)
- **Scale**: ~12.4M nuclei, ~45K genes, 74 embryos, 15 experimental batches
- **Developmental stages**: E8.0–P0 (gastrula to birth)
- **3 modalities**: total RNA (h5ad), exon/spliced (RDS), intron/unspliced (RDS)

**Data sources**:
- [Authors' download portal](https://omg.gs.washington.edu/jax/public/download.html)
- [CellxGene](https://cellxgene.cziscience.com/collections/45d5d2c3-bc28-4814-aed6-0bb6f0e11c82)
- [GEO GSE228590](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE228590)

**Goal**: Download all data, merge into a single AnnData with 3 layers (total, spliced, unspliced),
extract metadata (including `pcr_well` from barcodes), compute QC metrics, and save an
unfiltered checkpoint for downstream analysis.

In [ ]:
import gc
import glob
import os

import anndata as ad
import matplotlib
import matplotlib._text_helpers
import matplotlib.backends.backend_agg  # noqa: F401 — force-load before rds2py (known conflict)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc

from regularizedvi.utils import filter_genes

DATA_DIR = "/nfs/team205/vk7/sanger_projects/large_data/gastrula_to_pup"
os.makedirs(DATA_DIR, exist_ok=True)

## 1. Download data from authors' portal

The data is hosted at `shendure-web.gs.washington.edu` and consists of:

| Data | Files | Size | Format |
|------|-------|------|--------|
| Total RNA counts | `adata_JAX_dataset_{1,2,3,4}.h5ad` | ~42 GB | AnnData (h5ad) |
| Cell metadata | `df_cell.csv` | 1.5 GB | CSV |
| Gene metadata | `df_gene.csv` | 1.7 MB | CSV |
| Exon/intron counts | `embryo_{1..74}_exp_{exon,intron}.rds` | ~47 GB | R sparse matrices (RDS) |

The download uses `wget -c` for resume support (important for large files over slow connections).

In [ ]:
import subprocess

BASE_URL = "https://shendure-web.gs.washington.edu/content/members/cxqiu/public/backup/jax/download"

# --- h5ad files (total RNA) + metadata CSVs ---
adata_files = [
    "adata/adata_JAX_dataset_1.h5ad",
    "adata/adata_JAX_dataset_2.h5ad",
    "adata/adata_JAX_dataset_3.h5ad",
    "adata/adata_JAX_dataset_4.h5ad",
    "adata/df_cell.csv",
    "adata/df_gene.csv",
]

for f in adata_files:
    dest = os.path.join(DATA_DIR, os.path.basename(f))
    if not os.path.exists(dest):
        url = f"{BASE_URL}/{f}"
        print(f"Downloading {os.path.basename(f)}...")
        subprocess.run(["wget", "-c", url, "-O", dest], check=True)
    else:
        print(f"Already exists: {os.path.basename(f)}")

In [ ]:
# --- Exon/intron RDS files (148 files: 74 embryos x 2 modalities) ---
rds_dir = os.path.join(DATA_DIR, "embryo_exon_intron")
os.makedirs(rds_dir, exist_ok=True)

for i in range(1, 75):
    for modality in ["exon", "intron"]:
        fname = f"embryo_{i}_exp_{modality}.rds"
        dest = os.path.join(rds_dir, fname)
        if not os.path.exists(dest):
            url = f"{BASE_URL}/embryo_exon_intron/{fname}"
            print(f"Downloading {fname}...")
            subprocess.run(["wget", "-c", url, "-O", dest], check=True)

rds_files = sorted(glob.glob(os.path.join(rds_dir, "*.rds")))
print(f"\nTotal RDS files: {len(rds_files)}")

### Alternative: fast parallel download from the shell

For faster downloads (~90 GB total), run these commands directly in a terminal
instead of through Python:

```bash
# Set data directory
DATA_DIR="/nfs/team205/vk7/sanger_projects/large_data/gastrula_to_pup"
BASE_URL="https://shendure-web.gs.washington.edu/content/members/cxqiu/public/backup/jax/download"

# Download h5ad files + metadata CSVs (6 files in parallel, ~42 GB)
cd "$DATA_DIR"
for f in adata_JAX_dataset_1.h5ad adata_JAX_dataset_2.h5ad \
         adata_JAX_dataset_3.h5ad adata_JAX_dataset_4.h5ad \
         df_cell.csv df_gene.csv; do
    wget -c -q "${BASE_URL}/adata/${f}" -O "${f}" &
done
wait
echo "h5ad + metadata downloads complete"

# Download 148 exon/intron RDS files in batches of 20 (~47 GB)
mkdir -p "${DATA_DIR}/embryo_exon_intron"
cd "${DATA_DIR}/embryo_exon_intron"
for i in $(seq 1 74); do
    for mod in exon intron; do
        f="embryo_${i}_exp_${mod}.rds"
        [ ! -f "$f" ] && wget -c -q "${BASE_URL}/embryo_exon_intron/${f}" -O "${f}" &
    done
    if [ $((i % 10)) -eq 0 ]; then wait; echo "Downloaded up to embryo_${i}..."; fi
done
wait
echo "All RDS downloads complete"
```

## 2. Load total RNA h5ad files

The 4 h5ad files contain the total RNA count matrix split across chunks.
We load and concatenate them into a single AnnData.

In [ ]:
h5ad_paths = sorted(glob.glob(os.path.join(DATA_DIR, "adata_JAX_dataset_*.h5ad")))
print(f"Found {len(h5ad_paths)} h5ad files")

adatas = []
for path in h5ad_paths:
    print(f"Loading {os.path.basename(path)}...")
    adata_chunk = sc.read_h5ad(path)
    adata_chunk.X = adata_chunk.X.astype(np.uint16)  # uint16 before concat (counts ≤65535)
    adatas.append(adata_chunk)
    print(f"  {adata_chunk.shape}, dtype={adata_chunk.X.dtype}")

adata = ad.concat(adatas, join="outer", merge="same")
del adatas
gc.collect()
print(f"\nTotal RNA AnnData: {adata.shape}")
print(f"X dtype: {adata.X.dtype}")
print(f"var columns preserved: {list(adata.var.columns)}")
print(f"obs columns preserved: {list(adata.obs.columns)}")

# Strip dataset-chunk suffix from obs_names
# The h5ad files add a chunk-level suffix (-0 through -12) to each barcode, but the
# canonical cell_id (used in RDS files and df_cell.csv) has no suffix.
# Format: run_4_P2-01A.ATTCAAGCATGTTACGCAAG-0  ->  run_4_P2-01A.ATTCAAGCATGTTACGCAAG
raw_barcodes = adata.obs_names.str.replace(r"-\d+$", "", regex=True)
assert raw_barcodes.is_unique, f"Barcodes not unique after stripping suffix! {raw_barcodes.duplicated().sum()} dups"
adata.obs_names = raw_barcodes
print("\nStripped chunk suffix from obs_names (verified unique)")
print(f"Example: {adata.obs_names[0]}")

## 3. Merge cell metadata

Load `df_cell.csv` (cell annotations: embryo_id, day, experimental_batch, celltype, etc.)
and `df_gene.csv` (gene annotations) and merge into the AnnData.

In [ ]:
df_cell = pd.read_csv(os.path.join(DATA_DIR, "df_cell.csv"))
df_gene = pd.read_csv(os.path.join(DATA_DIR, "df_gene.csv"))

print(f"df_cell: {df_cell.shape}")
print(f"  columns: {list(df_cell.columns)}")
print(f"\ndf_gene: {df_gene.shape}")
print(f"  columns: {list(df_gene.columns)}")

# Map cell annotations to adata.obs using reindex (safer than join/merge)
# Only add columns not already present in adata.obs (h5ad already has day, embryo_id, etc.)
df_cell = df_cell.set_index("cell_id")
new_cols = [c for c in df_cell.columns if c not in adata.obs.columns and c != "Unnamed: 0"]
print(f"\nColumns already in adata.obs: {[c for c in df_cell.columns if c in adata.obs.columns]}")
print(f"New columns to add: {new_cols}")

if new_cols:
    for col in new_cols:
        adata.obs[col] = df_cell[col].reindex(adata.obs_names).values

print(f"\nAnnData obs columns after merge: {list(adata.obs.columns)}")
for col in new_cols:
    n_valid = adata.obs[col].notna().sum()
    print(f"  {col}: {n_valid:,} / {adata.n_obs:,} non-null ({100 * n_valid / adata.n_obs:.1f}%)")

## 4. Extract `pcr_well` from cell barcodes

sci-RNA-seq3 uses **3-level combinatorial indexing**. The cell barcode encodes the PCR well
(Level 3), which determines sequencing depth and protease-step ambient RNA.

Canonical barcode format: `run_4_P2-01A.ATTCAAGCATGTTACGCAAG`
- `run_4` = sequencing run (16 runs total across 15 experiments)
- `P2-01A` = PCR well (plate-row-column) → this is what we extract
- `ATTCAAGCATGTTACGCAAG` = combinatorial barcode (RT + ligation)

### Barcode alignment between h5ad and RDS files

The authors provide three data formats on their
[download portal](https://shendure-web.gs.washington.edu/content/members/cxqiu/public/backup/jax/download/):

| Format | Cell ID style | Source |
|--------|---------------|--------|
| h5ad (total RNA) | `run_4_P2-01A.BARCODE-N` (with chunk suffix `-0` to `-12`) | `adata/adata_JAX_dataset_{1..4}.h5ad` |
| RDS (exon/intron) | `run_4_P2-01A.BARCODE` (canonical, no suffix) | `embryo_exon_intron/` |
| df_cell.csv | `run_4_P2-01A.BARCODE` (canonical, no suffix) | `adata/df_cell.csv` |

The h5ad chunk suffix (`-0` through `-12`) is an artifact of splitting cells across 4 dataset
files — it maps 1:1 to sequencing runs but is **not** part of the biological cell identifier.
The canonical `cell_id` used in `df_cell.csv` and the per-embryo RDS matrices has no suffix.
We strip it above to align all three sources. After stripping, all 11.4M barcodes remain unique.

**References**:
- [Authors' analysis code (JAX_code)](https://github.com/ChengxiangQiu/JAX_code): the
  `doExtractData()` function in `JAX_help_code.R` reads per-experiment cell annotations
  using `cell_id` as the canonical identifier
- [sci-RNA-seq3 protocol](https://teichlab.github.io/scg_lib_structs/methods_html/sci-RNA-seq_family.html):
  3-level indexing (RT barcode + ligation/Tn5 barcode + PCR barcode)
- [Qiu et al. 2024, Nature 626:1084–1093](https://doi.org/10.1038/s41586-024-07069-w):
  Methods section describes 15 sci-RNA-seq3 experiments across 21 Novaseq runs

Multiple embryos (~12) share each PCR well. The PCR well determines:
- Sequencing budget allocation (`library_size_key`)
- Protease-step ambient contamination (`ambient_covariate_keys`)

In [ ]:
# Extract PCR well from cell barcode
# Format: run_X_PLATE-WELL.BARCODE
# Example: run_4_P2-01A.ATTCAAGCATGTTACGCAAG -> pcr_well = "P2-01A"
pattern = r"run_\d+_([^.]+)\."
adata.obs["pcr_well"] = adata.obs_names.str.extract(pattern, expand=False)

# Also extract run_id for reference
adata.obs["run_id"] = adata.obs_names.str.extract(r"(run_\d+)_", expand=False)

print(f"Unique pcr_wells: {adata.obs['pcr_well'].nunique()}")
print(f"Unique runs: {adata.obs['run_id'].nunique()}")
print(f"Unique embryos: {adata.obs['embryo_id'].nunique()}")
print(f"Unique experiments: {adata.obs['experimental_batch'].nunique()}")

# Cross-tabulation: how many embryos per PCR well?
print("\nEmbryos per PCR well:")
print(adata.obs.groupby("pcr_well")["embryo_id"].nunique().describe())

In [ ]:
# Create composite dispersion column
# dispersion_key only accepts a single str, so we create a composite of embryo_id x experimental_batch
adata.obs["embryo_experiment"] = adata.obs["embryo_id"].astype(str) + "_" + adata.obs["experimental_batch"].astype(str)
print(f"Unique embryo_experiment groups: {adata.obs['embryo_experiment'].nunique()}")

## 5. Gene identifiers

Check the gene identifier format and map using `df_gene.csv` if needed.

In [ ]:
# Inspect gene identifiers
print(f"First 5 var_names: {list(adata.var_names[:5])}")
print(f"var columns: {list(adata.var.columns)}")
print("\nFirst 5 rows of adata.var:")
print(adata.var.head())

# var_names are Ensembl IDs; gene_short_name column has symbols (from h5ad metadata)
if "gene_short_name" in adata.var.columns:
    print(f"\ngene_short_name mapped: {adata.var['gene_short_name'].notna().sum()} / {adata.n_vars}")
    mt_check = adata.var["gene_short_name"].str.startswith("mt-", na=False).sum()
    print(f"MT genes (mt- prefix): {mt_check}")
elif "gene_id" in df_gene.columns and "gene_short_name" in df_gene.columns:
    # Fallback: merge from df_gene if not already in var
    gene_map = df_gene.set_index("gene_id")
    for col in ["gene_short_name", "chr"]:
        if col not in adata.var.columns and col in gene_map.columns:
            adata.var[col] = adata.var_names.map(gene_map[col])
    print(f"Mapped from df_gene: {list(adata.var.columns)}")

## 6. Compute QC metrics

Compute standard QC metrics on the total RNA counts:
- Total UMI counts per cell
- Number of genes detected per cell
- Mitochondrial fraction (mouse: `mt-` or `Mt-` prefix)

In [ ]:
# Ensure counts are in .X
if "counts" in adata.layers:
    adata.X = adata.layers["counts"]

# Basic QC
sc.pp.calculate_qc_metrics(adata, inplace=True)

# Mitochondrial fraction
# var_names are Ensembl IDs; use gene_short_name or chr column for MT detection
if "gene_short_name" in adata.var.columns:
    mt_genes = adata.var["gene_short_name"].str.startswith("mt-", na=False)
elif "chr" in adata.var.columns:
    mt_genes = adata.var["chr"] == "chrM"
else:
    # Fallback: map chr from df_gene
    chr_map = df_gene.set_index("gene_id")["chr"]
    adata.var["chr"] = adata.var_names.map(chr_map)
    mt_genes = adata.var["chr"] == "chrM"

mt_counts = np.array(adata[:, mt_genes].X.sum(1)).squeeze()
adata.obs["mt_frac"] = mt_counts / adata.obs["total_counts"]

print(f"MT genes found: {mt_genes.sum()}")
print(f"MT fraction:\n{adata.obs['mt_frac'].describe()}")
print(f"\nTotal counts:\n{adata.obs['total_counts'].describe()}")
print(f"\nN genes:\n{adata.obs['n_genes_by_counts'].describe()}")

## 7. QC visualisation

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

qc_cols = ["total_counts", "n_genes_by_counts", "mt_frac"]
for i, col in enumerate(qc_cols):
    ax = axes.flat[i]
    if col in adata.obs.columns:
        adata.obs[col].hist(ax=ax, bins=100)
        ax.set_title(col)
        ax.set_ylabel("Cells")

# Cells per embryo
ax = axes[1, 0]
cells_per_embryo = adata.obs.groupby("embryo_id").size().sort_values()
ax.barh(range(len(cells_per_embryo)), cells_per_embryo.values)
ax.set_xlabel("Cells")
ax.set_title(f"Cells per embryo (n={len(cells_per_embryo)})")
ax.set_yticks([])

# Cells per PCR well (distribution, too many for individual bars)
ax = axes[1, 1]
cells_per_well = adata.obs.groupby("pcr_well").size()
ax.hist(cells_per_well.values, bins=50)
ax.set_xlabel("Cells per PCR well")
ax.set_ylabel("Number of wells")
ax.set_title(f"Cells per PCR well (n={len(cells_per_well)})")

# Cells per experimental_batch
ax = axes[1, 2]
cells_per_batch = adata.obs.groupby("experimental_batch").size().sort_values()
ax.barh(range(len(cells_per_batch)), cells_per_batch.values)
ax.set_yticks(range(len(cells_per_batch)))
ax.set_yticklabels(cells_per_batch.index, fontsize=8)
ax.set_xlabel("Cells")
ax.set_title(f"Cells per batch (n={len(cells_per_batch)})")

plt.tight_layout()
plt.show()

In [ ]:
# Metadata summary tables
print("Cells per embryo:")
print(adata.obs.groupby("embryo_id").size().describe())

print("\nCells per PCR well:")
print(adata.obs.groupby("pcr_well").size().describe())

print("\nCells per experimental_batch:")
print(adata.obs.groupby("experimental_batch").size().describe())

print("\nEmbryos per developmental day:")
print(adata.obs.groupby("day")["embryo_id"].nunique())

## 8. Save consolidated h5ad

Save the full RNA AnnData (with QC metrics in `.obs`) for use by the Scrublet notebook (NB1a)
and the MuData building notebook (NB1b).

In [ ]:
h5ad_path = os.path.join(DATA_DIR, "gastrula_to_pup_rna_qc.h5ad")
adata.write_h5ad(h5ad_path)
print(f"Saved {adata.shape} to {h5ad_path}")
print(f"File size: {os.path.getsize(h5ad_path) / 1e9:.1f} GB")
print(f"obs columns: {list(adata.obs.columns)}")

## 9. Gene filtering exploration

Preview gene filtering to evaluate thresholds before running the full MuData build (NB1b).
`filter_genes` shows a 2D histogram of gene detection rate vs mean expression, with filter
boundaries overlaid. Adjust `cell_count_cutoff`, `cell_percentage_cutoff2`, and
`nonz_mean_cutoff` as needed, then use the same thresholds in NB1b.

In [ ]:
from rds2py import read_rds

rds_dir = os.path.join(DATA_DIR, "embryo_exon_intron")

# Get shared genes between RNA (h5ad) and exon/intron (RDS)
_exon = read_rds(os.path.join(rds_dir, "embryo_1_exp_exon.rds"))
exon_var_names = pd.Index(list(_exon.dimnames[0]))
del _exon
gc.collect()

shared_genes = adata.var_names.intersection(exon_var_names)
print(f"RNA genes: {adata.n_vars}")
print(f"Exon/intron genes: {len(exon_var_names)}")
print(f"Shared genes: {len(shared_genes)}")
print(f"RNA-only genes (not in exon/intron): {adata.n_vars - len(shared_genes)}")

# Preview gene filtering on shared-gene subset
# filter_genes adds n_cells, nonz_mean to .var and shows 2D histogram
adata_shared = adata[:, shared_genes]  # view, no copy
selected = filter_genes(adata_shared, cell_count_cutoff=15, cell_percentage_cutoff2=0.01, nonz_mean_cutoff=1.03)
print(f"\nGenes passing filter: {len(selected)} / {len(shared_genes)} ({100 * len(selected) / len(shared_genes):.1f}%)")
print(f"Genes removed: {len(shared_genes) - len(selected)}")